# GCM BA - 2. Calibration of F1: EC-Earth3 historical vs ERA5 (EC-Earth3 grid, 0.70 deg)

**F1 = fixed WDF + QDM**, per month and per pixel (QDM: one xsdba training per calendar month; 50 quantiles for pr, 100 for tasmax and DTR), between ERA5 regridded onto the EC-Earth3 grid (reference) and
EC-Earth3 historical (to correct), over `CAL_PERIOD` (1981-2014), on the whole globe. Inputs read directly from the input files (checked by notebook 1).
`VARS_RUN = ["pr"]` calibrates a single variable first (to measure the computing time); the parameter files already
written are kept (`OVERWRITE = False`), so the other variables can be added later without recomputing it.

| variable | treatment |
|---|---|
| `pr` | **fixed WDF** (dry days), then multiplicative QDM on wet days |
| `tasmax` | additive QDM |
| `tasmin` | through the diurnal range `DTR = tasmax - tasmin` (multiplicative QDM); `tasmin = tasmax - DTR` (so tasmin < tasmax) |

**Fixed WDF rule**: estimated **once** here (per month and pixel: model threshold, activation probability of a dry day,
fixed seed) and applied **unchanged** to every run (notebook 3). The dry-day frequency of ERA5 is therefore never
re-imposed on a future / hosing run: their own dry-day signal is preserved.

```
elisa/GCM_BA/                           (EOS; inputs read directly from GCM_downscaling/Input_data/ and EC-Earth_hosing/)
├── 2_F1_calculation/    F1_<period>/       : WDF rule + QDM parameters
├── GCM_bias_adjusted_0.7deg/  <run>_F1_<period>.nc : historical + 4 GCM runs bias-adjusted (ERA5-like), EC-Earth3 grid
└── 4_verification/cache/                   : statistics of the verification
```

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_CORE = cfg["variables"]                                   # pr, tasmax, tasmin: required
VARS_ALL = VARS_CORE                # rsds not bias-adjusted (to add it back: VARS_CORE + cfg.get("extra_variables", []))
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]

CAL_PERIOD = ("1981", "2014")      # years of ERA5 / EC-Earth3 historical used for F1 (EC-Earth3 historical ends in 2014)
RSDS_FIRST_YEAR = "2005"           # ERA5 radiation starts in 2005: shorter period for rsds

def period_of(v):
    """Calibration period of a variable: CAL_PERIOD, starting in 2005 for rsds."""
    return (max(CAL_PERIOD[0], RSDS_FIRST_YEAR), CAL_PERIOD[1]) if v == "rsds" else CAL_PERIOD

F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}"
CHUNKS_SPACE = {"lat": 32, "lon": 64}   # dask spatial blocks (time = a single block, required by QDM)
INTERP = "linear"        # xsdba QDM interpolation for the in-sample check (same as notebook 3)
VARS_RUN = None          # None = all the variables | e.g. ["pr"] for a first test (tasmin needs tasmax)
OVERWRITE = False        # False = keep the parameter files already written
F1_DIR = bu.gcm_ba_dir(cfg, "2_F1_calculation", F1_TAG)
print("F1 parameters:", F1_DIR)

## 1. Calibration pair (read directly from the input files)

In [ ]:
WANTED = VARS_ALL if VARS_RUN is None else [v for v in VARS_ALL if v in VARS_RUN]
VARS = [v for v in WANTED if bu.gcm_ba_available(cfg, "era5", v) and bu.gcm_ba_available(cfg, "gcm_historical", v)]
assert VARS == WANTED, f"inputs missing for {set(WANTED) - set(VARS)}: check the config (notebook 00) and notebook 1"
ref = {v: bu.gcm_ba_input(cfg, "era5", v, period_of(v)) for v in VARS}            # ERA5 regridded with CDO
hist = {v: bu.gcm_ba_input(cfg, "gcm_historical", v, period_of(v)) for v in VARS}  # EC-Earth3 historical
for v in VARS:
    assert bu.same_grid(ref[v], hist[v])
    print(f"{v:7s} {period_of(v)[0]}-{period_of(v)[1]} | ERA5 {ref[v].sizes['time']} days | EC-Earth3 {hist[v].sizes['time']} days")

## 2. Calibration (fixed WDF, then QDM)

In [ ]:
import time
t0 = time.time()
bu.fit_F(ref, hist, F1_DIR, chunks_space=CHUNKS_SPACE, overwrite=OVERWRITE)
print(f"F1 calibration ({', '.join(VARS)}): {(time.time() - t0) / 60:.1f} min")
info_path = F1_DIR / "f1_info.json"
info = json.loads(info_path.read_text()) if info_path.exists() else {"variables": [], "periods": {}}
info["variables"] = [v for v in VARS_ALL if v in set(info["variables"]) | set(VARS)]
info["periods"].update({v: list(period_of(v)) for v in VARS})
info.update({"grid": "EC-Earth3 gr", "mask": "none (whole globe)", "inputs": {"ref": bu.GCM_BA_ERA5, "hist": "gcm_historical"}})
info_path.write_text(json.dumps(info, indent=2))
print(sorted(p.name for p in F1_DIR.iterdir()))

## 3. The WDF rule (if `pr` was calibrated)

`p0_ref` / `p0_hist`: dry-day fraction of ERA5 / of EC-Earth3; `t`: threshold applied to EC-Earth3 (above 0.1 mm/day if the
model is too wet); `q`: probability of activating a dry day of EC-Earth3 (if the model is too dry).

In [ ]:
assert (F1_DIR / "wdf_params.nc").exists(), "pr not calibrated yet: no WDF rule to show"
par = xr.open_dataset(F1_DIR / "wdf_params.nc").load()
fig, axes = plt.subplots(2, 4, figsize=(20, 7))
for r, m in enumerate([1, 7]):
    for c, (k, kw) in enumerate([("p0_ref", dict(vmin=0, vmax=1)), ("p0_hist", dict(vmin=0, vmax=1)),
                                 ("t", dict(vmin=0.1, vmax=1.0)), ("q", dict(vmin=0, vmax=0.5))]):
        a = par[k].sel(month=m)
        im = axes[r, c].pcolormesh(a.lon, a.lat, a, shading="auto", **kw); plt.colorbar(im, ax=axes[r, c], shrink=0.7)
        axes[r, c].set_facecolor("0.85"); axes[r, c].set_title(f"{k} - month {m}")
plt.tight_layout(); plt.savefig(F1_DIR / "wdf_rule.png", dpi=110); plt.show()

## 4. In-sample check (first 3 years)

F1 applied to EC-Earth3 historical itself: its quantiles should fall back onto those of ERA5. Same period as the
training, so this only checks the code; the real checks are in notebook 4.

In [ ]:
sl = slice(0, 365 * 3)
H = {v: x.isel(time=sl) for v, x in hist.items()}
A = xr.Dataset(bu.apply_F(H, F1_DIR, chunks_space=CHUNKS_SPACE, interp=INTERP)).compute()
rows = []
for v in [v for v in VARS if v in A]:
    for lab, x in (("ERA5 (target)", ref[v].isel(time=sl).compute()), ("raw EC-Earth3", H[v].compute()), ("EC-Earth3 + F1", A[v])):
        s = bu.summarize(x)
        rows.append({"var": v, "series": lab, **{k: round(val, 3) for k, val in s.items() if k != "n"}})
display(pd.DataFrame(rows).set_index(["var", "series"]))